# ISK vs fondkonto baseline

This notebook answers the narrow baseline question: if the entire fondkonto is liquidated today, taxes are paid immediately, and the proceeds are repurchased directly inside the existing ISK, how does that compare with keeping the holdings in the fondkonto?

The goal is not full precision yet. The goal is to build a defensible first-pass baseline with editable tax assumptions.

In [ ]:
from pathlib import Path
import sys

import pandas as pd
import plotly.express as px

candidate_dirs = [Path.cwd(), Path.cwd() / 'copilot_analysis']
analysis_dir = next(path for path in candidate_dirs if (path / 'fund_analysis.py').exists())
if str(analysis_dir) not in sys.path:
    sys.path.append(str(analysis_dir))

from fund_analysis import AccountComparisonInputs, DEFAULT_PORTFOLIO, TaxAssumptions, compare_accounts, compare_accounts_grid

In [2]:
assumptions = {
    'market_value': DEFAULT_PORTFOLIO['market_value'],
    'cost_basis': DEFAULT_PORTFOLIO['cost_basis'],
    'existing_isk_value': DEFAULT_PORTFOLIO['existing_isk_value'],
    'annual_return': 0.07,
    'years': 30,
    'tax': TaxAssumptions(
        capital_gains_tax_rate=0.30,
        isk_tax_rate=0.01065,
        isk_tax_free_amount=300000.0,
    ),
}
assumptions

{'market_value': 773517.0,
 'cost_basis': 181574.76,
 'existing_isk_value': 630000.0,
 'annual_return': 0.07,
 'years': 30,
 'tax': TaxAssumptions(capital_gains_tax_rate=0.3, isk_tax_rate=0.01065, isk_tax_free_amount=300000.0)}

In [3]:
baseline_result = compare_accounts(AccountComparisonInputs(**assumptions))
baseline_table = pd.DataFrame([baseline_result]).T.rename(columns={0: 'value'})
baseline_table

,value
investable_isk_capital,5.959343e+05
tax_paid_now,1.775827e+05
isk_tax_free_amount,3.000000e+05
taxable_isk_base_marginal,5.959343e+05
annual_isk_tax,6.346701e+03
taxable_isk_base_total_account,9.259343e+05
annual_isk_tax_total_account,9.861201e+03
isk_terminal_value,4.346003e+06
fondkonto_terminal_value,4.176219e+06
difference_isk_minus_fondkonto,1.697846e+05


In [4]:
horizon_grid = compare_accounts_grid(
    market_value=assumptions['market_value'],
    cost_basis=assumptions['cost_basis'],
    existing_isk_value=assumptions['existing_isk_value'],
    annual_returns=[assumptions['annual_return']],
    years=range(1, 41),
    tax=assumptions['tax'],
)
horizon_grid.head()

,annual_return,years,investable_isk_capital,tax_paid_now,isk_tax_free_amount,taxable_isk_base_marginal,annual_isk_tax,taxable_isk_base_total_account,annual_isk_tax_total_account,isk_terminal_value,fondkonto_terminal_value,difference_isk_minus_fondkonto
0,0.07,1.0,595934.328,177582.672,300000.0,595934.328,6346.700593,925934.328,9861.200593,631303.030367,633836.661000,-2533.630633
1,0.07,2.0,595934.328,177582.672,300000.0,595934.328,6346.700593,925934.328,9861.200593,669591.810941,674392.157310,-4800.346369
2,0.07,3.0,595934.328,177582.672,300000.0,595934.328,6346.700593,925934.328,9861.200593,711005.075197,717786.538362,-6781.463165
3,0.07,4.0,595934.328,177582.672,300000.0,595934.328,6346.700593,925934.328,9861.200593,755761.536992,764218.526087,-8456.989095
4,0.07,5.0,595934.328,177582.672,300000.0,595934.328,6346.700593,925934.328,9861.200593,804095.220154,813900.752953,-9805.532799


In [5]:
fig = px.line(
    horizon_grid,
    x='years',
    y='difference_isk_minus_fondkonto',
    # y=["isk_terminal_value", "fondkonto_terminal_value"],
    title='ISK minus fondkonto by horizon',
    labels={'difference_isk_minus_fondkonto': 'SEK advantage', 'years': 'Years'}
)
fig.add_hline(y=0, line_dash='dash')
fig.show()

## Interpretation checklist

Use this notebook to lock down the baseline assumptions before moving on to the broader break-even grid. If the baseline logic changes, update `fund_analysis.py` first and keep the notebooks thin.